# 1. Data Loading
Load H&M Dataset

In [1]:
import pandas as pd
import os

In [30]:
# Load the article metadata
DATA_DIR = "/Users/chen/Desktop/Nus/Class/Practice Module/Project/fashion-recommender/data/raw/"
ARTICLES_PATH = os.path.join(DATA_DIR, "articles.csv")
CUSTOMERS_PATH = os.path.join(DATA_DIR, "customers.csv")
TRANSACTIONS_PATH = os.path.join(DATA_DIR, "transactions_train.csv")

# Ensure IDs are strings for safe joins
articles_df = pd.read_csv(ARTICLES_PATH, dtype={"article_id": "string"})
customers = pd.read_csv(CUSTOMERS_PATH, dtype={"customer_id": "string"})
transactions = pd.read_csv(
    TRANSACTIONS_PATH,
    dtype={"customer_id": "string", "article_id": "string"},
    parse_dates=["t_dat"],
)

print("articles:", articles_df.shape)
print("customers:", customers.shape)
print("transactions:", transactions.shape)


articles: (105542, 25)
customers: (1371980, 7)
transactions: (31788324, 5)


# 2. Top–Bottom Filtering
Filter articles into tops and bottoms using the confirmed `product_type_name` lists, then restrict transactions accordingly.

In [31]:
ladies_top_df = articles_df[
    (articles_df['product_group_name'] == 'Garment Upper body') &
    (articles_df['index_name'] == 'Ladieswear')
].copy()
ladies_bottom_df = articles_df[
    (articles_df['product_group_name'] == 'Garment Lower body') &
    (articles_df['index_name'] == 'Ladieswear')
].copy()

ladies_top_ids = set(ladies_top_df['article_id'])
ladies_bottom_ids = set(ladies_bottom_df['article_id'])
allowed_ids = ladies_top_ids.union(ladies_bottom_ids)

ladies_relevant_transactions = transactions[transactions['article_id'].isin(allowed_ids)].copy()
ladies_customer_retained  = ladies_transactions_filtered['customer_id'].nunique()

print("tops:", ladies_top_df.shape)
print("bottoms:", ladies_bottom_df.shape)
print("filtered transactions:", ladies_transactions_filtered.shape)
print("retained customers:", ladies_customer_retained)
print("Unique tops:")

tops: (14095, 25)
bottoms: (5701, 25)
filtered transactions: (10761918, 5)
retained customers: 992849
Unique tops:


In [ ]:
# Tag each row as 'Top' or 'Bottom' based on the article_id
ladies_relevant_transactions['type'] = ladies_relevant_transactions['article_id'].apply(
    lambda x: 'Top' if x in ladies_top_ids else 'Bottom' if x in ladies_bottom_ids else 'Unknown'
)


 Found 5312044 co-occurrences of tops and bottoms bought on the same day by the same customer.


In [26]:
# Define Min occurance
min_cooccurrence = 100

# Count how many times each (top, bottom) pair occurs
relation_counts = merged.groupby(['article_id_top', 'article_id_bottom']).size().reset_index(name='weight')

# Filter by threshold (Remove weak connections)
strong_relations = relation_counts[relation_counts['weight'] >= min_cooccurrence]

# Add the relation name
strong_relations['relation'] = 'matches_with'

# Reorder for KG format : head, relation, tail, weight
kg_edges = strong_relations[['article_id_top', 'relation', 'article_id_bottom', 'weight']].copy()

print (f"Generated {len(kg_edges)} strong edges for the knowledge graph.")

# Save
OUTPUT_DIR = "/Users/chen/Desktop/Nus/Class/Practice Module/Project/fashion-recommender/data/processed/kg/top_bottom_relation.csv"
kg_edges.to_csv(OUTPUT_DIR, index=False)
print(f"Saved KG edges to {OUTPUT_DIR}")

Generated 484 strong edges for the knowledge graph.
Saved KG edges to /Users/chen/Desktop/Nus/Class/Practice Module/Project/fashion-recommender/data/processed/kg/top_bottom_relation.csv


# Part 5 Define Logic Function

In [32]:
def get_weather(row):
    " Determine weather nased on keywords"
    desc = str(row['detail_desc']).lower()
    prod_type = str(row['product_type_name']).lower()
    
    # Cold / Winter
    if any(x in desc for x in ['wool', 'fleece', 'knit', 'thermal', 'padded', 'warm']) or \
        prod_type in ['coat', 'jacket', 'sweater', 'hoodie', 'cardigan']:
        return 'Cold'

    # Hot / Summer
    if any(x in desc for x in ['linen', 'sleeveless', 'straps', 'short sleeve', 'coolmax']) or \
        prod_type in ['t-shirt', 'tank top', 'shorts', 'skirt', 'dress', 'bikini', 'swimsuit', 'swinwear']:
        return 'Hot'
    
    # Default if no strong signal
    return 'Mild'

def get_occasion(row):
    " Determine occasion based on keywords"
    desc = str(row['detail_desc']).lower()
    prod_type = str(row['product_type_name']).lower()
    section = str(row['section_name']).lower()

    # Sport / Activewear
    if 'sport' in section or prod_type in ['leggings', 'sports bra', 'running shorts', 'activewear']:
        return 'Sport'
    
    # Formal / Offixw
    if prod_type in ['blazer', 'suit', 'trousers', 'shirt', 'blouse'] or \
        'silk' in desc or 'tailored' in desc:
        return 'Formal'

    # Party / Night Out
    if any(x in desc for x in ['sequin', 'lace', 'velvet', 'satin', 'metallic']) or \
        'party' in section:
        return 'Party'

    # Casual / Everyday
    return 'Casual'

Apply Logic and Generate Edge

In [34]:
# Apply the functions to create new columns
articles_df['weather_node'] = articles_df.apply(get_weather, axis=1)
articles_df['occasion_node'] = articles_df.apply(get_occasion, axis=1)

# Create the "Weather" Edges (Item -> Suitable_for -> Weather)
weather_edges = articles_df[['article_id', 'weather_node']].copy()
weather_edges.columns = ['head', 'tail']
weather_edges['relation'] = 'suitable_for_weather'

# Create the "Occasion" Edges (Item -> Suitable_for -> Occasion)
occasion_edges = articles_df[['article_id', 'occasion_node']].copy()
occasion_edges.columns = ['head', 'tail']
occasion_edges['relation'] = 'suitable_for_occasion'

# Combine them into one DataFrame
attributes_kg = pd.concat([weather_edges, occasion_edges])

# Reorder coloumns to match final kf format: head, relation, tail
attributes_kg = attributes_kg[['head', 'relation', 'tail']]

print(f"Generated {len(attributes_kg)} attribute edges for the knowledge graph.")

Generated 211084 attribute edges for the knowledge graph.


In [39]:
final_graph = pd.concat([kg_edges, attributes_kg], ignore_index=True)

# Save
OUTPUT_DIR = "/Users/chen/Desktop/Nus/Class/Practice Module/Project/fashion-recommender/data/processed/kg/final_kg.csv"
final_graph.to_csv(OUTPUT_DIR, index=False)
print(f"Saved final KG to {OUTPUT_DIR}")

Saved final KG to /Users/chen/Desktop/Nus/Class/Practice Module/Project/fashion-recommender/data/processed/kg/final_kg.csv
